# 07 — Linear Regression From Scratch

In the previous notebook, we learned how **Autograd** computes gradients automatically.

Now we will use those ideas to build and train our first complete machine-learning model:

> **Linear Regression**

Linear regression is one of the best models for learning the complete training process because it contains almost every important idea used later in neural networks:

- Parameters
- Predictions
- Loss functions
- Gradients
- Gradient descent
- Training loops
- Learning rates
- Model evaluation

## In this notebook, we will learn:

1. Linear regression intuition
2. The equation $y=wx+b$
3. Synthetic data
4. Prediction / forward pass
5. Mean squared error
6. Manual gradients
7. Autograd gradients
8. Gradient descent
9. Training loop
10. Learning rate
11. Loss curves
12. Comparing manual training with `nn.Linear`
13. Common training mistakes
14. Debugging a model that does not learn
15. Practice exercises

## Main Goal

By the end of this notebook, you should understand this complete learning cycle:

$$
\text{Data}
\rightarrow
\text{Prediction}
\rightarrow
\text{Loss}
\rightarrow
\text{Gradients}
\rightarrow
\text{Parameter Update}
\rightarrow
\text{Better Prediction}
$$

We will first build everything ourselves before using PyTorch's higher-level abstractions.


In [ ]:
import torch
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)


# 1. What Is Linear Regression?

Linear regression tries to model the relationship between an input $x$ and an output $y$ using a straight line.

The basic equation is:

$$
\boxed{y=wx+b}
$$

where:

$$
\begin{array}{|c|c|}
\hline
\textbf{Symbol} & \textbf{Meaning} \\
\hline
x & \text{Input feature} \\
\hline
w & \text{Weight / slope} \\
\hline
b & \text{Bias / intercept} \\
\hline
y & \text{Predicted output} \\
\hline
\end{array}
$$

The model learns the values of:

- $w$
- $b$

from data.


# 2. Understanding the Equation $y=wx+b$

Suppose:

$$
w=2
$$

and:

$$
b=1
$$

Then:

$$
y=2x+1
$$

For different inputs:

$$
\begin{array}{|c|c|}
\hline
x & y=2x+1 \\
\hline
0 & 1 \\
\hline
1 & 3 \\
\hline
2 & 5 \\
\hline
3 & 7 \\
\hline
4 & 9 \\
\hline
\end{array}
$$

The weight controls how steeply the output changes.

The bias shifts the line up or down.


In [ ]:
x = torch.tensor([0., 1., 2., 3., 4.])

w = 2.0
b = 1.0

y = w * x + b

print("x:", x)
print("y:", y)


# 3. Weight as Slope

For:

$$
y=wx+b
$$

the weight $w$ is the slope.

If:

$$
w>0
$$

the line rises as $x$ increases.

If:

$$
w<0
$$

the line falls as $x$ increases.

If:

$$
w=0
$$

the prediction does not change with $x$.

For example:

$$
y=3x+1
$$

increases by `3` whenever $x$ increases by `1`.


# 4. Bias as Intercept

The bias $b$ determines the prediction when:

$$
x=0
$$

because:

$$
y=w(0)+b=b
$$

So the bias is the point where the line crosses the $y$-axis.

Example:

$$
y=2x+5
$$

At:

$$
x=0
$$

we get:

$$
y=5
$$


# 5. Creating Synthetic Data

To understand training clearly, we will create our own dataset from known parameters.

Let the true relationship be:

$$
\boxed{y=3x+2}
$$

Therefore:

$$
w_{true}=3
$$

and:

$$
b_{true}=2
$$

We will create input values and generate targets from this equation.


In [ ]:
torch.manual_seed(42)

x = torch.linspace(-5, 5, 100)

true_w = 3.0
true_b = 2.0

y = true_w * x + true_b

print("x shape:", x.shape)
print("y shape:", y.shape)

print("\\nFirst 5 inputs:", x[:5])
print("First 5 targets:", y[:5])


The dataset follows a perfect straight-line relationship.

In real datasets, measurements usually contain noise.

Let's add a small amount of random noise so the example is more realistic.


In [ ]:
torch.manual_seed(42)

noise = torch.randn_like(y) * 1.0
y_noisy = y + noise

print("First 5 noisy targets:", y_noisy[:5])


# 6. Visualizing the Synthetic Data

The points should lie roughly around the true line:

$$
y=3x+2
$$


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(x.numpy(), y_noisy.numpy(), label="Observed data", alpha=0.7)
plt.plot(x.numpy(), y.numpy(), label="True line")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic Linear Regression Data")
plt.legend()
plt.show()


# 7. Train and Test Split

We should not evaluate a model only on the same data used for training.

We will divide the dataset into:

- Training data
- Test data

For this simple tutorial:

- First 80 samples → training
- Last 20 samples → test


In [ ]:
split_index = 80

x_train = x[:split_index]
y_train = y_noisy[:split_index]

x_test = x[split_index:]
y_test = y_noisy[split_index:]

print("Training samples:", x_train.shape[0])
print("Test samples:", x_test.shape[0])


# 8. Starting With Bad Parameters

The true values are:

$$
w=3
$$

$$
b=2
$$

But the model does not know them.

We will deliberately start with poor guesses:

$$
w=0
$$

$$
b=0
$$

Training should gradually move these parameters toward useful values.


In [ ]:
w = torch.tensor(0.0)
b = torch.tensor(0.0)

print("Initial weight:", w)
print("Initial bias:", b)


# 9. The Forward Pass

A **forward pass** means using the current model parameters to make predictions.

Our model is:

$$
\hat{y}=wx+b
$$

We use $\hat{y}$ to distinguish the prediction from the true target $y$.


In [ ]:
def predict(x, w, b):
    return w * x + b

predictions = predict(x_train, w, b)

print("First 5 predictions:", predictions[:5])


Because we started with:

$$
w=0,\qquad b=0
$$

every prediction is initially:

$$
\hat{y}=0
$$

The predictions are poor.

Now we need a way to measure **how wrong** they are.


# 10. Error and Residuals

For each sample, the prediction error is:

$$
e_i=\hat{y}_i-y_i
$$

Example:

If:

$$
y=10
$$

and:

$$
\hat{y}=7
$$

then:

$$
e=7-10=-3
$$

The difference between observed and predicted values is often called a **residual**.


In [ ]:
errors = predictions - y_train

print("First 5 errors:", errors[:5])


# 11. Mean Squared Error

A common loss function for regression is **Mean Squared Error (MSE)**.

For $N$ samples:

$$
\boxed{
MSE=
\frac{1}{N}
\sum_{i=1}^{N}
(\hat{y}_i-y_i)^2
}
$$

The steps are:

1. Prediction minus target
2. Square the error
3. Average all squared errors

Squaring is useful because:

- Positive and negative errors do not cancel
- Larger errors are penalized more strongly


In [ ]:
def mse_loss(predictions, targets):
    return ((predictions - targets) ** 2).mean()

loss = mse_loss(predictions, y_train)

print("Initial MSE:", loss.item())


# 12. MSE Step by Step

Consider:

$$
y=
\begin{array}{|c|c|c|}
\hline
2 & 4 & 6 \\
\hline
\end{array}
$$

and predictions:

$$
\hat{y}=
\begin{array}{|c|c|c|}
\hline
1 & 5 & 4 \\
\hline
\end{array}
$$

Errors:

$$
\begin{array}{|c|c|c|}
\hline
-1 & 1 & -2 \\
\hline
\end{array}
$$

Squared errors:

$$
\begin{array}{|c|c|c|}
\hline
1 & 1 & 4 \\
\hline
\end{array}
$$

Therefore:

$$
MSE=
\frac{1+1+4}{3}
=
\boxed{2}
$$


In [ ]:
targets = torch.tensor([2., 4., 6.])
preds = torch.tensor([1., 5., 4.])

errors = preds - targets
squared_errors = errors ** 2
mse = squared_errors.mean()

print("Errors:", errors)
print("Squared errors:", squared_errors)
print("MSE:", mse)


# 13. What Training Must Do

Our objective is:

> Find values of $w$ and $b$ that minimize the loss.

The current parameters produce predictions.

The predictions produce a loss.

We need to know:

$$
\frac{\partial L}{\partial w}
$$

and:

$$
\frac{\partial L}{\partial b}
$$

These gradients tell us how the loss changes when we change the parameters.


# 14. Deriving the Gradient Manually

For one sample:

$$
\hat{y}=wx+b
$$

and squared error:

$$
L=(\hat{y}-y)^2
$$

Substitute the prediction:

$$
L=(wx+b-y)^2
$$

For the weight:

$$
\frac{\partial L}{\partial w}
=
2(wx+b-y)x
$$

For the bias:

$$
\frac{\partial L}{\partial b}
=
2(wx+b-y)
$$

For mean squared error across $N$ samples:

$$
\boxed{
\frac{\partial L}{\partial w}
=
\frac{2}{N}
\sum_{i=1}^{N}
(\hat{y}_i-y_i)x_i
}
$$

and:

$$
\boxed{
\frac{\partial L}{\partial b}
=
\frac{2}{N}
\sum_{i=1}^{N}
(\hat{y}_i-y_i)
}
$$


# 15. Manual Gradient Computation

Let's calculate these gradients ourselves.


In [ ]:
w = torch.tensor(0.0)
b = torch.tensor(0.0)

predictions = predict(x_train, w, b)
errors = predictions - y_train

dw = 2 * (errors * x_train).mean()
db = 2 * errors.mean()

print("Manual dw:", dw.item())
print("Manual db:", db.item())


# 16. Gradient Descent

Gradient descent updates parameters in the **opposite direction of the gradient**.

For the weight:

$$
\boxed{
w_{new}
=
w_{old}
-
\eta
\frac{\partial L}{\partial w}
}
$$

For the bias:

$$
\boxed{
b_{new}
=
b_{old}
-
\eta
\frac{\partial L}{\partial b}
}
$$

where:

$$
\eta
$$

is the **learning rate**.


# 17. One Manual Gradient-Descent Step

Let's perform one update manually.


In [ ]:
learning_rate = 0.01

w = torch.tensor(0.0)
b = torch.tensor(0.0)

predictions = predict(x_train, w, b)
loss_before = mse_loss(predictions, y_train)

errors = predictions - y_train

dw = 2 * (errors * x_train).mean()
db = 2 * errors.mean()

w = w - learning_rate * dw
b = b - learning_rate * db

predictions_after = predict(x_train, w, b)
loss_after = mse_loss(predictions_after, y_train)

print("Loss before:", loss_before.item())
print("Loss after :", loss_after.item())
print("Updated w:", w.item())
print("Updated b:", b.item())


If the learning rate is reasonable, the loss should decrease after the update.

That is the central idea of optimization:

> Repeatedly update the parameters so the loss becomes smaller.


# 18. Manual Training Loop

Now we repeat:

1. Forward pass
2. Loss calculation
3. Gradient calculation
4. Parameter update

many times.


In [ ]:
w_manual = torch.tensor(0.0)
b_manual = torch.tensor(0.0)

learning_rate = 0.01
epochs = 200

manual_losses = []

for epoch in range(epochs):
    # 1. Forward pass
    predictions = predict(x_train, w_manual, b_manual)

    # 2. Loss
    loss = mse_loss(predictions, y_train)

    # 3. Manual gradients
    errors = predictions - y_train
    dw = 2 * (errors * x_train).mean()
    db = 2 * errors.mean()

    # 4. Update parameters
    w_manual = w_manual - learning_rate * dw
    b_manual = b_manual - learning_rate * db

    manual_losses.append(loss.item())

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"w: {w_manual.item():.4f} | "
            f"b: {b_manual.item():.4f}"
        )


The learned parameters should move close to the true values:

$$
w_{true}=3
$$

$$
b_{true}=2
$$

Because the data contains noise, the learned values may not be exactly equal to them.


In [ ]:
print("True w:", true_w)
print("Learned w:", w_manual.item())

print()

print("True b:", true_b)
print("Learned b:", b_manual.item())


# 19. Plotting the Manual Training Loss

A loss curve helps us see whether training is improving.

A healthy training run often shows the loss decreasing over time.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(manual_losses)
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("Manual Gradient Descent — Training Loss")
plt.show()


# 20. Visualizing the Learned Line

Let's compare:

- Observed data
- True line
- Learned line


In [ ]:
with torch.no_grad():
    learned_line = predict(x, w_manual, b_manual)

plt.figure(figsize=(8, 5))
plt.scatter(x.numpy(), y_noisy.numpy(), label="Observed data", alpha=0.7)
plt.plot(x.numpy(), y.numpy(), label="True line")
plt.plot(x.numpy(), learned_line.numpy(), label="Learned line")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Manual Linear Regression")
plt.legend()
plt.show()


# 21. Evaluating on the Test Set

We trained using only the training samples.

Now evaluate the learned parameters on the test data.

During evaluation, gradients are not needed.


In [ ]:
with torch.no_grad():
    test_predictions = predict(x_test, w_manual, b_manual)
    test_loss = mse_loss(test_predictions, y_test)

print("Test MSE:", test_loss.item())


# 22. Linear Regression With Autograd

Now we will train the same model again, but this time PyTorch will calculate the gradients automatically.

We need:

`requires_grad=True`

for trainable parameters.


In [ ]:
w_auto = torch.tensor(0.0, requires_grad=True)
b_auto = torch.tensor(0.0, requires_grad=True)

print("w requires_grad:", w_auto.requires_grad)
print("b requires_grad:", b_auto.requires_grad)


# 23. One Autograd Training Step

The steps are:

1. Forward pass
2. Loss
3. `loss.backward()`
4. Update parameters
5. Clear gradients


In [ ]:
learning_rate = 0.01

predictions = predict(x_train, w_auto, b_auto)
loss = mse_loss(predictions, y_train)

loss.backward()

print("Loss:", loss.item())
print("dw:", w_auto.grad.item())
print("db:", b_auto.grad.item())


Now update the parameters.

The update should not itself become part of the computational graph.

Therefore we use:

`torch.no_grad()`


In [ ]:
with torch.no_grad():
    w_auto -= learning_rate * w_auto.grad
    b_auto -= learning_rate * b_auto.grad

print("Updated w:", w_auto.item())
print("Updated b:", b_auto.item())


Now clear the accumulated gradients.


In [ ]:
w_auto.grad.zero_()
b_auto.grad.zero_()

print("w.grad:", w_auto.grad)
print("b.grad:", b_auto.grad)


# 24. Full Autograd Training Loop

Now combine everything into a training loop.


In [ ]:
w_auto = torch.tensor(0.0, requires_grad=True)
b_auto = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.01
epochs = 200

autograd_losses = []

for epoch in range(epochs):
    # 1. Forward pass
    predictions = predict(x_train, w_auto, b_auto)

    # 2. Compute loss
    loss = mse_loss(predictions, y_train)

    # 3. Backward pass
    loss.backward()

    # 4. Update parameters
    with torch.no_grad():
        w_auto -= learning_rate * w_auto.grad
        b_auto -= learning_rate * b_auto.grad

    # 5. Clear gradients
    w_auto.grad.zero_()
    b_auto.grad.zero_()

    autograd_losses.append(loss.item())

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"w: {w_auto.item():.4f} | "
            f"b: {b_auto.item():.4f}"
        )


# 25. Comparing Manual Gradients With Autograd

For the same parameters and data, manual gradients and Autograd gradients should agree.


In [ ]:
w_test = torch.tensor(1.5, requires_grad=True)
b_test = torch.tensor(0.5, requires_grad=True)

predictions = predict(x_train, w_test, b_test)
loss = mse_loss(predictions, y_train)

# Manual gradients
errors = predictions.detach() - y_train
dw_manual = 2 * (errors * x_train).mean()
db_manual = 2 * errors.mean()

# Autograd gradients
loss.backward()

print("Manual dw  :", dw_manual.item())
print("Autograd dw:", w_test.grad.item())

print()

print("Manual db  :", db_manual.item())
print("Autograd db:", b_test.grad.item())


The values should be extremely close.

This confirms that Autograd is performing the calculus we derived manually.


# 26. Why We Clear Gradients

PyTorch accumulates gradients.

If we forget:

`w.grad.zero_()`

and:

`b.grad.zero_()`

the next backward pass adds new gradients to the previous gradients.

That changes the optimization behavior.


In [ ]:
w_demo = torch.tensor(1.0, requires_grad=True)

loss1 = (w_demo - 5) ** 2
loss1.backward()

print("After first backward:", w_demo.grad.item())

loss2 = (w_demo - 5) ** 2
loss2.backward()

print("After second backward:", w_demo.grad.item())


In ordinary training loops, clear gradients once per optimization step.

Later, optimizers will provide:

`optimizer.zero_grad()`


# 27. Understanding the Learning Rate

The learning rate controls the **size of each parameter update**.

$$
parameter_{new}
=
parameter_{old}
-
learning\_rate \times gradient
$$

If the learning rate is too small:

- Training may be very slow

If the learning rate is too large:

- Training may overshoot
- Loss may oscillate
- Loss may increase
- Training may become unstable

A good learning rate makes steady progress.


# 28. Comparing Learning Rates

Let's train the same model using several learning rates.


In [ ]:
def train_with_learning_rate(lr, epochs=100):
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)

    losses = []

    for _ in range(epochs):
        predictions = predict(x_train, w, b)
        loss = mse_loss(predictions, y_train)

        loss.backward()

        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad

        w.grad.zero_()
        b.grad.zero_()

        losses.append(loss.item())

    return losses, w.item(), b.item()

small_losses, small_w, small_b = train_with_learning_rate(0.001)
good_losses, good_w, good_b = train_with_learning_rate(0.01)
large_losses, large_w, large_b = train_with_learning_rate(0.1)

print("lr=0.001:", small_w, small_b)
print("lr=0.01 :", good_w, good_b)
print("lr=0.1  :", large_w, large_b)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(small_losses, label="lr = 0.001")
plt.plot(good_losses, label="lr = 0.01")
plt.plot(large_losses, label="lr = 0.1")
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("Effect of Learning Rate")
plt.legend()
plt.yscale("log")
plt.show()


# 29. What Should We Look for in a Loss Curve?

A loss curve can reveal training problems.

## Healthy Training

Typically:

$$
Loss \downarrow
$$

and eventually begins to level off.

## Too-Small Learning Rate

Loss decreases very slowly.

## Too-Large Learning Rate

Loss may:

- Jump around
- Increase
- Become extremely large
- Become `nan`

The loss curve is one of the first debugging tools you should inspect.


# 30. Building the Model With `nn.Linear`

Until now, we manually stored:

- `w`
- `b`

PyTorch provides a built-in linear layer:

`nn.Linear`

For one input feature and one output feature:

`nn.Linear(1, 1)`


In [ ]:
import torch.nn as nn

model = nn.Linear(in_features=1, out_features=1)

print(model)


# 31. Understanding `nn.Linear(1, 1)`

PyTorch stores:

$$
weight.shape=(1,\ 1)
$$

and:

$$
bias.shape=(1)
$$

The model computes a linear transformation equivalent to:

$$
\hat{y}=wx+b
$$

for this one-dimensional example.


In [ ]:
print("Weight shape:", model.weight.shape)
print("Bias shape:", model.bias.shape)

print("Initial weight:", model.weight)
print("Initial bias:", model.bias)


# 32. Input Shape for `nn.Linear`

`nn.Linear` expects the final dimension to contain the features.

Our current training input has shape:

$$
(80)
$$

We want:

$$
(80,\ 1)
$$

meaning:

- 80 samples
- 1 feature per sample

Use:

`unsqueeze(1)`


In [ ]:
X_train = x_train.unsqueeze(1)
Y_train = y_train.unsqueeze(1)

X_test = x_test.unsqueeze(1)
Y_test = y_test.unsqueeze(1)

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)


# 33. Built-In MSE Loss

PyTorch provides:

`nn.MSELoss()`

This performs mean squared error.


In [ ]:
criterion = nn.MSELoss()

example_prediction = model(X_train)
example_loss = criterion(example_prediction, Y_train)

print("Prediction shape:", example_prediction.shape)
print("Loss:", example_loss.item())


# 34. Using an Optimizer

Instead of manually updating weights, PyTorch provides optimizers.

For gradient descent, we can use:

`torch.optim.SGD`

The optimizer handles parameter updates for us.


In [ ]:
import torch.optim as optim

model = nn.Linear(1, 1)

criterion = nn.MSELoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.01
)

print(optimizer)


# 35. Standard PyTorch Training Loop

A standard training loop looks like:

1. `optimizer.zero_grad()`
2. Forward pass
3. Compute loss
4. `loss.backward()`
5. `optimizer.step()`

The order is important.


In [ ]:
torch.manual_seed(42)

model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 200
nn_losses = []

for epoch in range(epochs):
    # 1. Clear old gradients
    optimizer.zero_grad()

    # 2. Forward pass
    predictions = model(X_train)

    # 3. Loss
    loss = criterion(predictions, Y_train)

    # 4. Backward pass
    loss.backward()

    # 5. Update parameters
    optimizer.step()

    nn_losses.append(loss.item())

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1:03d} | "
            f"Loss: {loss.item():.4f}"
        )


# 36. Inspecting the Learned `nn.Linear` Parameters

Let's inspect what the model learned.


In [ ]:
learned_weight = model.weight.item()
learned_bias = model.bias.item()

print("True weight:", true_w)
print("Learned weight:", learned_weight)

print()

print("True bias:", true_b)
print("Learned bias:", learned_bias)


The values should be close to the true relationship:

$$
y=3x+2
$$

Again, because the targets include random noise, the best-fit parameters may not be exactly `3` and `2`.


# 37. Evaluating the `nn.Linear` Model

During evaluation:

- We do not update parameters
- We do not need gradients

So use:

`torch.no_grad()`


In [ ]:
with torch.no_grad():
    test_predictions_nn = model(X_test)
    test_loss_nn = criterion(test_predictions_nn, Y_test)

print("Test MSE:", test_loss_nn.item())


# 38. Visualizing the `nn.Linear` Model


In [ ]:
with torch.no_grad():
    full_predictions_nn = model(x.unsqueeze(1)).squeeze(1)

plt.figure(figsize=(8, 5))
plt.scatter(x.numpy(), y_noisy.numpy(), label="Observed data", alpha=0.7)
plt.plot(x.numpy(), y.numpy(), label="True line")
plt.plot(x.numpy(), full_predictions_nn.numpy(), label="nn.Linear prediction")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear Regression With nn.Linear")
plt.legend()
plt.show()


# 39. Comparing the Three Approaches

We have now trained linear regression in three ways:

1. Manual gradients
2. Autograd gradients
3. `nn.Linear` + optimizer

All three solve the same mathematical problem.

$$
\begin{array}{|c|c|}
\hline
\textbf{Approach} & \textbf{What PyTorch Handles} \\
\hline
\text{Manual gradients} & \text{Almost nothing} \\
\hline
\text{Autograd} & \text{Gradient calculation} \\
\hline
nn.Linear + optimizer & \text{Parameters + gradients + updates} \\
\hline
\end{array}
$$

Understanding all three levels makes higher-level PyTorch code much easier to understand.


# 40. Comparing the Loss Curves


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(manual_losses, label="Manual gradients")
plt.plot(autograd_losses, label="Autograd")
plt.plot(nn_losses, label="nn.Linear + SGD")
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("Comparing Training Methods")
plt.legend()
plt.show()


# 41. Why the Curves May Not Be Identical

The manual and Autograd versions began from exactly:

$$
w=0,\qquad b=0
$$

The `nn.Linear` model uses its own random initialization by default.

Therefore its early training trajectory can differ.

However, all approaches should move toward a good solution.


# 42. Model Parameters

For any PyTorch model, you can inspect trainable parameters with:

`model.parameters()`

or:

`model.named_parameters()`


In [ ]:
for name, parameter in model.named_parameters():
    print(name)
    print("shape:", parameter.shape)
    print("value:", parameter.data)
    print("gradient:", parameter.grad)
    print()


This is an important debugging habit.

Later, neural networks may have many layers and thousands or millions of parameters.


# 43. Prediction vs Target Shapes

A very common regression mistake is mismatched shapes.

For example:

Prediction:

$$
(80,\ 1)
$$

Target:

$$
(80)
$$

PyTorch may broadcast these tensors in a way you did not intend.

It is better to ensure both have matching shapes.

For our model:

$$
prediction.shape=(80,\ 1)
$$

and:

$$
target.shape=(80,\ 1)
$$


In [ ]:
with torch.no_grad():
    predictions = model(X_train)

print("Prediction shape:", predictions.shape)
print("Target shape:", Y_train.shape)


# 44. Common Training Mistakes

## Mistake 1 — Forgetting to Clear Gradients

Gradients accumulate unless you clear them.

With an optimizer:

`optimizer.zero_grad()`

## Mistake 2 — Forgetting `backward()`

Without:

`loss.backward()`

the parameter gradients are not computed.

## Mistake 3 — Forgetting `optimizer.step()`

Without:

`optimizer.step()`

the parameters never change.

## Mistake 4 — Wrong Learning Rate

Too small → training is slow.

Too large → training can become unstable.

## Mistake 5 — Shape Mismatch

Always inspect:

- Input shape
- Prediction shape
- Target shape

## Mistake 6 — Updating During Evaluation

Do not train on validation or test data.

## Mistake 7 — Tracking Gradients During Inference

Use:

`torch.no_grad()`

when gradients are not required.

## Mistake 8 — Calling `.item()` Too Early

Do not convert a loss to a Python number before calling `backward()`.

This is wrong:

`loss.item().backward()`

The loss must remain a tensor for Autograd.


# 45. Debugging a Model That Does Not Learn

If the loss is not decreasing, inspect these things:

1. Are the inputs and targets correct?
2. Are input and target shapes correct?
3. Are parameters receiving gradients?
4. Is the learning rate reasonable?
5. Are gradients being cleared?
6. Is `backward()` being called?
7. Is `optimizer.step()` being called?
8. Are parameters actually changing?
9. Is the loss function appropriate?
10. Is the data itself learnable?


In [ ]:
for name, parameter in model.named_parameters():
    print(
        name,
        "| value:",
        parameter.data.flatten().tolist(),
        "| grad:",
        None if parameter.grad is None else parameter.grad.flatten().tolist()
    )


# 46. A Minimal PyTorch Training Template

This pattern will appear repeatedly throughout deep learning.


In [ ]:
# Generic training-loop structure

# for epoch in range(num_epochs):
#     model.train()
#
#     optimizer.zero_grad()
#
#     predictions = model(inputs)
#
#     loss = criterion(predictions, targets)
#
#     loss.backward()
#
#     optimizer.step()


Memorizing the code is less important than understanding why each line exists.

$$
\begin{array}{|c|c|}
\hline
\textbf{Line} & \textbf{Purpose} \\
\hline
optimizer.zero_grad() & \text{Clear old gradients} \\
\hline
model(inputs) & \text{Forward pass} \\
\hline
criterion(...) & \text{Measure prediction error} \\
\hline
loss.backward() & \text{Compute gradients} \\
\hline
optimizer.step() & \text{Update parameters} \\
\hline
\end{array}
$$


# 47. Practice Exercises

Try solving these before looking at the solutions.

## Exercise 1

For:

$$
y=4x+3
$$

calculate $y$ when:

$$
x=5
$$

## Exercise 2

If:

$$
w=2
$$

$$
b=-1
$$

and:

$$
x=
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
\end{array}
$$

calculate the predictions.

## Exercise 3

For targets:

$$
\begin{array}{|c|c|c|}
\hline
2 & 4 & 6 \\
\hline
\end{array}
$$

and predictions:

$$
\begin{array}{|c|c|c|}
\hline
1 & 5 & 4 \\
\hline
\end{array}
$$

calculate MSE manually.

## Exercise 4

Create synthetic data using:

$$
y=5x-2
$$

## Exercise 5

Initialize:

$$
w=0,\qquad b=0
$$

and implement one manual gradient-descent step.

## Exercise 6

Train linear regression using manual gradients.

## Exercise 7

Train the same problem using Autograd.

## Exercise 8

Train the same problem using:

`nn.Linear(1,1)`

and:

`torch.optim.SGD`

## Exercise 9

Experiment with learning rates:

- `0.0001`
- `0.001`
- `0.01`
- `0.1`

Compare their loss curves.

## Exercise 10

Explain why test data should not be used to update model parameters.


# 48. Shape Reasoning Challenges

Answer before running code.

## Challenge 1

If:

$$
X.shape=(100)
$$

what shape should it have before passing it into:

`nn.Linear(1,1)`?

## Challenge 2

If:

$$
X.shape=(32,\ 5)
$$

and:

`nn.Linear(5, 3)`

what is the output shape?

## Challenge 3

For:

`nn.Linear(5,3)`

what are:

- `weight.shape`
- `bias.shape`

## Challenge 4

If predictions have shape:

$$
(64,\ 1)
$$

but targets have shape:

$$
(64)
$$

why might this be dangerous?

## Challenge 5

What five operations make up the standard training step?


# 49. Exercise Solutions


In [ ]:
# Exercise 1
x_ex1 = 5
y_ex1 = 4 * x_ex1 + 3
print("Exercise 1:", y_ex1)

# Exercise 2
x_ex2 = torch.tensor([1., 2., 3.])
w_ex2 = 2.0
b_ex2 = -1.0
print("Exercise 2:", w_ex2 * x_ex2 + b_ex2)

# Exercise 3
targets_ex3 = torch.tensor([2., 4., 6.])
preds_ex3 = torch.tensor([1., 5., 4.])
print("Exercise 3:", ((preds_ex3 - targets_ex3) ** 2).mean())

# Exercise 4
x_ex4 = torch.linspace(-2, 2, 20)
y_ex4 = 5 * x_ex4 - 2
print("Exercise 4 shapes:", x_ex4.shape, y_ex4.shape)

# Exercise 5
w_ex5 = torch.tensor(0.0)
b_ex5 = torch.tensor(0.0)

pred_ex5 = w_ex5 * x_train + b_ex5
err_ex5 = pred_ex5 - y_train

dw_ex5 = 2 * (err_ex5 * x_train).mean()
db_ex5 = 2 * err_ex5.mean()

lr_ex5 = 0.01

w_ex5 = w_ex5 - lr_ex5 * dw_ex5
b_ex5 = b_ex5 - lr_ex5 * db_ex5

print("Exercise 5 w:", w_ex5.item())
print("Exercise 5 b:", b_ex5.item())


# 50. Key Takeaways

In this notebook, we learned:

- Linear regression intuition
- The equation $y=wx+b$
- Weight and bias
- Synthetic data
- Train/test splitting
- Forward passes
- Residuals
- Mean squared error
- Manual gradients
- Gradient descent
- Manual training loops
- Autograd training
- Gradient clearing
- Learning rates
- Loss curves
- `nn.Linear`
- `nn.MSELoss`
- `torch.optim.SGD`
- Standard PyTorch training loops
- Model evaluation
- Common training mistakes

The complete training cycle is:

$$
\boxed{
\text{zero gradients}
\rightarrow
\text{forward}
\rightarrow
\text{loss}
\rightarrow
\text{backward}
\rightarrow
\text{update}
}
$$

This same structure will appear again and again as our models become more complex.


# 51. Check Your Understanding

Before moving to the next notebook, make sure you can answer these without searching:

1. What does linear regression try to learn?
2. What do $w$ and $b$ represent?
3. What is a forward pass?
4. What is a residual?
5. What is mean squared error?
6. Why do we square prediction errors?
7. What does a gradient tell us?
8. Why does gradient descent move opposite to the gradient?
9. What is a learning rate?
10. What happens if the learning rate is too small?
11. What can happen if it is too large?
12. Why must gradients be cleared?
13. Why are manual updates performed inside `torch.no_grad()`?
14. What does `nn.Linear(1,1)` represent?
15. What does `optimizer.zero_grad()` do?
16. What does `loss.backward()` do?
17. What does `optimizer.step()` do?
18. Why should test data not update model parameters?
19. What should a healthy loss curve generally look like?
20. How are manual gradients, Autograd, and `nn.Linear` related?


# Next Notebook

# 08 — Neural Network Foundations

In the next notebook, we will study:

- What is a neuron?
- Inputs, weights, and bias
- Linear transformation
- Why linear models are limited
- Activation functions
- ReLU
- Sigmoid
- Tanh
- Hidden layers
- Output layers
- Building a tiny neural network with raw tensors
- Building the same network with `nn.Module`
- Forward propagation
- Parameter counting
- Shape reasoning through layers
